In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import json
from scipy.stats import trim_mean
import math
from pandas.api.types import ( is_numeric_dtype, is_categorical_dtype, is_object_dtype, is_datetime64_any_dtype )
from default_risk.scripts.variable_profiling import eda_per_table_printing_results
from default_risk.scripts.variable_profiling import eda_per_table_persisting_result_html
from default_risk.scripts.variable_profiling import create_files_nulls_per_colmun
import default_risk.config as cfg
from default_risk.scripts.auxiliar_eda_function import check_invariant
import dtale
import logging

log = logging.getLogger('werkzeug')

bureau_df= pd.read_csv(cfg.BUREAU)

data_frame_size=len(bureau_df)

with open(cfg.SCHEMA_JSON, "r") as f:
    schema = json.load(f)

Invariants found at the moment: (# number of cell with the proofs and relevant code)

1- The factical enndate always point to the past: DAYS_CREDIT_ENDDATE < 0 (100%) #5


Soft constraints:

1- if have a factical end date defined ("DAYS_ENDDATE_FACT" != null), the loan is not active ("CREDICT_ACTIVE" != "Active") (99.9%) #3
Anomalies: Potential data corruption. 

2- if the contract are marked as closed ("CREDICT_ACTIVE" == "Closed") then the conctract have defined a factical end date ("DAYS_ENDDATE_FACT" != null) (99.99%)

Decisions summary: 

1- In cases where the soft constraint #1 is violated, we will input the status as "Closed" taking the DAYS_ENDDATE_FACT as source of true.

2- In cases where the soft contraint #2 we will drop the observations asumming data corruption.

In [ ]:
#1
#create the files por data data dictionary
create_files_nulls_per_colmun(bureau_df,"bureu")

In [ ]:
#2
#run the screening script on bureau
eda_per_table_printing_results(bureau_df, schema, "bureau",False)

In [62]:
#3
check_invariant((bureau_df["DAYS_ENDDATE_FACT"] > 0),"the factical enndate is positive (future date)",data_frame_size)

active_mask= (bureau_df["CREDIT_ACTIVE"] == "Active")
have_endate_mask= (bureau_df["DAYS_ENDDATE_FACT"].notna())
check_invariant((active_mask & have_endate_mask),"we have defined a factical date of end of contract but it's flagged as active",data_frame_size)

is_closed_mask= (bureau_df["CREDIT_ACTIVE"] == "Closed")
check_invariant((is_closed_mask & (~have_endate_mask)),"where are closed without factical endate",data_frame_size)

0 of cases where the factical enndate is positive (future date)
that represent a 0.0% of cases with violation of this invariant 

1969 of cases where we have defined a factical date of end of contract but it's flagged as active
that represent a 0.1147149778493476% of cases with violation of this invariant 

125 of cases where where are closed without factical endate
that represent a 0.0072825658868300915% of cases with violation of this invariant 



In [ ]:
#4
#in order to understand the nulls in "AMT_ANNUITY"
bureau_prev_contract_without_annuity= bureau_df[bureau_df["AMT_ANNUITY"].isnull()]
len(bureau_prev_contract_without_annuity)
rows_to_analyze=bureau_prev_contract_without_annuity[bureau_prev_contract_without_annuity["CREDIT_ACTIVE"] == "Active"]
print(rows_to_analyze["CREDIT_TYPE"].value_counts())
#seems like active loans of all kind can have AMT_ANNUITY as missing value. 
#This suggest that missing values are more correlated with the way the data are gathering than with the nature of the loans in the sample.

In [ ]:
#5
#in order to understand the missing values in "DAYS_ENDDATE_FACT"
without_endate= bureau_df[bureau_df["DAYS_ENDDATE_FACT"].isnull()]
len(without_endate)
print("status of observations with enndate in null:")
print(without_endate["CREDIT_ACTIVE"].value_counts())
print("status of all the dataset:")
print(bureau_df["CREDIT_ACTIVE"].value_counts())
#we have 125 rows without endate_fact but marked as closed. 



status of observations with enndate in null:
CREDIT_ACTIVE
Active      628638
Sold          4879
Closed         125
Bad debt        11
Name: count, dtype: int64
status of all the dataset:
CREDIT_ACTIVE
Closed      1079273
Active       630607
Sold           6527
Bad debt         21
Name: count, dtype: int64


1

In [ ]:
have_no_limit_card_mask= (bureau_df["AMT_CREDIT_SUM_LIMIT"].isnull()) 
credit_card_loan_mask= (bureau_df["CREDIT_TYPE"] == "Credit card")
is_active_mask= (bureau_df["CREDIT_ACTIVE"] == "Active")
rows_to_analyze= bureau_df[have_no_limit_card_mask & credit_card_loan_mask & is_active_mask]


print("we have " + str((credit_card_loan_mask & is_active_mask).sum()) + " active credit card loans")
print("where " + str(len(rows_to_analyze)) + " have no limit defined")

dtale.show(rows_to_analyze)
#this observation don't let clear the reason to exist credit card loans without limit but was useful to realize there is loans like 
#SK_ID_BUREAU = 5714465 where AMT_CREDIT_SUM is 0 and AMT_CREDIT_SUM_DEBT == 0 or nan at the same time. And with those 2 fields without providing info we don't have how to figured out the ammount of 
#the debt. This expose the need of define a minimun ammount of info provided for a observation to be considered in this table due the poor quality of the data.

we have 285998 active credit card loans
where 66256 have no limit defined


2026-05-09 22:43:38,762 - ERROR    - Exception occurred while processing request: object of type 'NoneType' has no len()
Traceback (most recent call last):
  File "c:\Users\kuroc\OneDrive\Escritorio\default risk\default-risk\Lib\site-packages\dtale\views.py", line 120, in _handle_exceptions
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\kuroc\OneDrive\Escritorio\default risk\default-risk\Lib\site-packages\dtale\views.py", line 1573, in get_processes
    [_load_process(data_id) for data_id in global_state.keys()],
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\kuroc\OneDrive\Escritorio\default risk\default-risk\Lib\site-packages\dtale\views.py", line 1573, in <listcomp>
    [_load_process(data_id) for data_id in global_state.keys()],
     ^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\kuroc\OneDrive\Escritorio\default risk\default-risk\Lib\site-packages\dtale\views.py", line 1558, in _load_process
    rows=len(data),
       

In [ ]:

col = bureau_df["AMT_CREDIT_SUM_LIMIT"]
#checking if there is very high values different that the max value (possible sentinel)
mask = (col > 30000) & (col != 31199.0)
mask2 = (col > 10950)
#checking if the ammout of values that will get clipped if i cap the column in 30 years
no_that_long = mask2.sum() - mask.sum()
print(no_that_long)
print(mask.sum())

In [ ]:
mask_sentinel_negative = ( col > 0 ) & (col > 10)
mask_sentinel_negative.sum()
